### RUN THE CLOSED-LOOP 

In [ ]:
# Common standard python libraries that are needed
import numpy as np
import logging
import os
import pandas as pd
from datetime import datetime, timedelta
import ast
import sys
import time

In [ ]:
# Custom Python functions that are needed
# cd path_to_agriacodm
# git clone https://github.com/DaveCacci/B.Agri_AcoDM.git
# sys.path.insert(0, r"path_to_agriacodm")
# cd path_to_twin_library
# git clone https://github.com/DaveCacci/B.Twin.git
# sys.path.insert(0, r"path_to_twin_library")
from test_closedloop_integrator import main as integrator_main # Uncomment only if you want to test a real closed-loop with the B.Agri_AcoDM as real plant simulator
from RPi_selectorPI_operative import main as main_selector

In [ ]:
# Define the number of closed-loop days you want to simulate
days_of_simulation = 45
control_interval = 6 # in hours (to be coherent with the values inside integrator.json)
control_steps = int(days_of_simulation*24/control_interval)
original_directory = os.getcwd()

# DECLARE LOGGER
# Configure logging
logger = logging.getLogger()
# Clean up any existing handlers for this logger
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
    handler.close()
# Console handler for displaying logs in the terminal
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_formatter = logging.Formatter('%(levelname)s - %(message)s')
console_handler.setFormatter(console_formatter)
# Add handlers to the logger
logger.addHandler(console_handler)
logger.setLevel(logging.INFO)

# When testing SelectorPI I must set the now time
current_timestamp = datetime(2025, 5, 30, 10) #datetime(2025, 5, 7, 13) #13:00 at index=0
control_steps = 2
for i in range(control_steps-1):
    os.chdir(original_directory)

    integrator_main() # Uncomment only if you want to test a real closed-loop with the B.Agri_AcoDM as real plant simulator
    os.chdir(original_directory)
    
    # When testing SelectorPI I must set the now time
    main_selector('1', now = current_timestamp)
    current_timestamp += timedelta(hours=control_interval)   
    os.chdir(original_directory)

    # Sleep for the duration of the control interval
    print('-------------------------------------Sleeping for 1 seconds...-----------------------------------------')
    time.sleep(1)  # Sleep for the specified interval (in seconds)

### POST-PROCESS OF RESULTS

In [ ]:
os.chdir("C:\\Users\\lenovo\\OneDrive - Politecnico di Milano\\Work_cloud\\DOTTORATO\\Sperimentazione control\\NMPC")

In [ ]:
# Common standard python libraries that are needed
import numpy as np
import logging
import os
import pandas as pd
from datetime import datetime, timedelta
import ast
import sys
sys.path.append("C:\\Users\\lenovo\\OneDrive - Politecnico di Milano\\Work_cloud\\DOTTORATO\\Sperimentazione MPC\\NMPC")

# Common functions from the 'MPC control' library
from general_utils.read_file import*
from general_utils.save_df import*
from general_utils.udm_gas_conversion import udm_gas_conversion
from general_utils.create_dataframe import create_dataframe
from general_utils.FlexiblePlotter import*

In [ ]:
testname = 'ClosedLooptest_tube_Real' #'ClosedLooptest_tube_rpi' #ClosedLooptest_tube_Real'
modelname = 'papercheck'
x_actual = read_csv_file(os.path.join(os.getcwd(), f"{testname}", f"{modelname}_y_df_off.txt"), log=True)
y_actual = read_csv_file(os.path.join(os.getcwd(), f"{testname}", f"{modelname}_y_df_on.txt"), log=True)
y_actual['qM_gb'] = y_actual['gas_rate']*y_actual['xch4_interp']
y_actual['co2ch4_gb'] = y_actual['xco2_interp']/y_actual['xch4_interp']
#
x_openloop = read_csv_file(os.path.join(os.getcwd(), f"{testname}", "Input", f"{modelname}_Openloop_model.csv"), log=True)
x_ekf = read_csv_file(os.path.join(os.getcwd(), f"{testname}", "Input", f"{modelname}_EKF.csv"), log=True)
P_ekf = read_csv_file(os.path.join(os.getcwd(), f"{testname}", "Input", f"{modelname}_EKF_diagCOV.csv"), log=True)
z_star_nom = read_csv_file(os.path.join(os.getcwd(), f"{testname}", f"{modelname}_nmpc_z_star_NOMINAL.csv"), log=True) #f"NMPC_R{modelname}_z_star_NOMINAL.csv")
osa_pred = read_csv_file(os.path.join(os.getcwd(), f"{testname}", f"{modelname}_Openloop_model_osaprediction.csv"), log=True)
# 
u_actual = read_csv_file(os.path.join(os.getcwd(), f"{testname}", "Input", f"{modelname}_nmpc_u_actual_ANCILLARY.csv"), log=True)
v_star_nom = read_csv_file(os.path.join(os.getcwd(), f"{testname}", "Input", f"{modelname}_nmpc_v_star_NOMINAL.csv"), log=True)
# 
y_star_nom = read_csv_file(os.path.join(os.getcwd(), f"{testname}", f"{modelname}_nmpc_y_star_NOMINAL.csv"), log=True)
# LOAD SETPOINTS FROM THE OFFLINE OPTIMIZATION MADE WITH AGRI-ACODM
y_ref = read_and_split_txt(os.path.join(os.getcwd(), f"{testname}", "Input", 'Output_setpoints.txt'), log=True)
y_ref = create_dataframe(['Qch4_ref', 'co2ch4_ref'], [pd.Series(y_ref[1]), pd.Series(y_ref[2])], datetime(2025,4,30,10,0))
#x_ref = read_and_split_txt(os.path.join(os.getcwd(), f"{testname}", "Input", 'State_setpoints.txt'), log=True)
# x_ref = create_dataframe(['Xh_ref', 'X1_ref', 'X2_ref', 'S1_ref', 'S2_ref', 'csi_ref','xMgb_ref'], [pd.Series(x_ref[1]), 
#                                                                                                     pd.Series(x_ref[2]),
#                                                                                                     pd.Series(x_ref[3]),
#                                                                                                     pd.Series(x_ref[4]),
#                                                                                                     pd.Series(x_ref[5]),
#                                                                                                     pd.Series(x_ref[6]),
#                                                                                                     pd.Series(x_ref[7]),
#                                                                                                 ], datetime(2025,4,30,10,0))
# Convert UdM to mmol/L/day
y_ref['Qch4_ref'] = udm_gas_conversion(y_ref['Qch4_ref'], (42+273.15), 1.035, 12, 'Lh')
#y_ref['S2_ref'] = x_ref['S2_ref']

In [ ]:
# Filter from star to end timestamp the dataframe
y_star_nom = y_star_nom[(y_star_nom['Timestamp'] >= datetime(2025, 5, 7, 10, 0)) & (y_star_nom['Timestamp'] < datetime(2025, 7, 10, 0, 0))]
y_ref = y_ref[(y_ref['Timestamp'] >= datetime(2025, 5, 7, 10, 0)) & (y_ref['Timestamp'] < datetime(2025, 7, 10, 0, 0))]
# Compute the error between the setpoints and the y_star_nom
y_star_nom.reset_index(drop=True, inplace=True)
y_ref.reset_index(drop=True, inplace=True)
y_star_nom['Qch4_ref'] = y_ref['Qch4_ref']
y_star_nom['co2ch4_ref'] = y_ref['co2ch4_ref']
y_star_nom['Qch4_ref'] = y_star_nom['Qch4_ref'].astype(float)
y_star_nom['co2ch4_ref'] = y_star_nom['co2ch4_ref'].astype(float)
y_star_nom['Qch4_error'] = y_star_nom['Qch4_ref'] - y_star_nom['qM_gb']
y_star_nom['co2ch4_error'] = y_star_nom['co2ch4_ref'] - y_star_nom['co2ch4_gb']
# Compute the Mean Absolute Relative Error (MARE)
y_star_nom['Qch4_error'] = y_star_nom['Qch4_error'].abs() / y_star_nom['Qch4_ref']*100
y_star_nom['co2ch4_error'] = y_star_nom['co2ch4_error'].abs() / y_star_nom['co2ch4_ref']*100

In [ ]:
# Compute the average of the Uk column in v_star_nom
v_star_nom = v_star_nom[(v_star_nom['Timestamp'] >= datetime(2025, 5, 7, 10, 0)) & (v_star_nom['Timestamp'] < datetime(2025, 7, 10, 0, 0))]
v_star_nom['Uk'] = v_star_nom['Uk'].astype(float)
v_star_nom['Uk'] = v_star_nom['Uk'].abs()   
v_star_nom['Uk'].mean()

In [ ]:
y_star_nom['Qch4_error'].mean(), y_star_nom['co2ch4_error'].mean()

In [ ]:
# # Read a csv separated with commas (from Modeica csv export)
# u_actual = pd.read_csv(os.path.join(os.getcwd(), f"{testname}", "Input", f"{modelname}_nmpc_u_actual_ANCILLARY_onmodelica.csv"))
# u_actual['Timestamp'] = datetime(2025,5,7,10) + pd.to_timedelta(u_actual['time'], unit='d')
# u_actual.insert(0, 'Timestamp', u_actual.pop('Timestamp'))

In [ ]:
plot_name = '_case_papercheck'

In [ ]:
#------------------------------------------------------------------------------------------------------------#
# PLOT
# Create interative plot! <<<<<<<<<<<<<<<<<<<<<<<<<
# Create an instance of FlexiblePlotter
plotter = FlexiblePlotter(default_figsize = (12, 4))

data = {"data_off_all": x_actual,
         "ekf": x_ekf,
         "model": x_openloop,
         "z_star": z_star_nom,
         #"osa_pred": osa_pred,
        }

# Dynamically create variable_groups as a dictionary
variable_groups = {key: ['S2'] for key in data.keys()}
variable_groups['data_off_all'] = ['TVFA']

# Calculate the x_limits
end_time = datetime(2025,6,30,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[-1]) #datetime.now()
start_time = datetime(2025,5,7,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[0])
x_limit = (start_time, end_time)
date_string = f'{start_time.strftime("%d%m%H")}-{end_time.strftime("%d%m%H")}'

# Dynamically generate a colors dictionary with lists of colors
keys = list(data.keys())
colormap = plt.cm.get_cmap('tab10', len(keys))  # Use 'tab10' or any other colormap
colors = {key: [colormap(i)] for i, key in enumerate(keys)}  # Wrap each color in a list

# Use the FlexiblePlotter to create the plot
plotter.plot_multiple_variables_across_dfs_with_labels(
    data=data,
    variables=variable_groups,
    x_axis_format="datetime",
    x_limit=x_limit,
    x_axis_ticks_format="%d-%m %H",
    tick_rotation=90,
    interval=7*12, #hours of tick interval x
    grid=True,
    show_plot=False,
    #y_limit=(0, 100),
    plot_types={'data_on':['line'], 'data_off_all':['line'], 'from_rpi':['line'],'rpi_data':['line'], 'osaprediction':['scatter']},
    colors=colors,
)
# output_path = os.path.join(os.path.dirname(input_path), 'awite_plots')
output_path = os.path.join(os.getcwd(), 'ClosedLooptest_tube_Real', 'plots_postprocess\\')

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the figure
#plt.savefig(os.path.join(output_path, f"1_{date_string}_S1{plot_name}.png"), bbox_inches='tight', dpi=300)

# Log the output path
logging.info(f"Plot saved to {output_path}")
#------------------------------------------------------------------------------------------------------------# 

In [ ]:
#------------------------------------------------------------------------------------------------------------#
# PLOT
# Create interative plot! <<<<<<<<<<<<<<<<<<<<<<<<<
# Create an instance of FlexiblePlotter
plotter = FlexiblePlotter(default_figsize = (12, 4))

data = {"data_off_all": x_actual,
         "ekf": x_ekf,
         "model": x_openloop,
         "z_star": z_star_nom,
         #"osa_pred": osa_pred,
        }

# Dynamically create variable_groups as a dictionary
variable_groups = {key: ['csi'] for key in data.keys()}
variable_groups['data_off_all'] = ['Alk-Sic']

# Calculate the x_limits
end_time = datetime(2025,6,30,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[-1]) #datetime.now()
start_time = datetime(2025,5,7,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[0])
x_limit = (start_time, end_time)
date_string = f'{start_time.strftime("%d%m%H")}-{end_time.strftime("%d%m%H")}'

# Dynamically generate a colors dictionary with lists of colors
keys = list(data.keys())
colormap = plt.cm.get_cmap('tab10', len(keys))  # Use 'tab10' or any other colormap
colors = {key: [colormap(i)] for i, key in enumerate(keys)}  # Wrap each color in a list

# Use the FlexiblePlotter to create the plot
plotter.plot_multiple_variables_across_dfs_with_labels(
    data=data,
    variables=variable_groups,
    x_axis_format="datetime",
    x_limit=x_limit,
    x_axis_ticks_format="%d-%m %H",
    tick_rotation=90,
    interval=7*12, #hours of tick interval x
    grid=True,
    show_plot=False,
    y_limit=(-10, 10),
    plot_types={'data_on':['line'], 'data_off_all':['line'], 'from_rpi':['line'],'rpi_data':['line'], 'osaprediction':['scatter']},
    colors=colors,
)
# output_path = os.path.join(os.path.dirname(input_path), 'awite_plots')
output_path = os.path.join(os.getcwd(), 'ClosedLooptest_tube_Real', 'plots_postprocess\\')

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the figure
#plt.savefig(os.path.join(output_path, f"1_{date_string}_S1{plot_name}.png"), bbox_inches='tight', dpi=300)

# Log the output path
logging.info(f"Plot saved to {output_path}")
#------------------------------------------------------------------------------------------------------------# 

In [ ]:
#------------------------------------------------------------------------------------------------------------#
# PLOT
# Create interative plot! <<<<<<<<<<<<<<<<<<<<<<<<<
# Create an instance of FlexiblePlotter
plotter = FlexiblePlotter(default_figsize = (12, 4))

data = {"u_actual": u_actual,
        "v_star": v_star_nom,
        }

# Dynamically create variable_groups as a dictionary
variable_groups = {key: ['Uk'] for key in data.keys()}

# Calculate the x_limits
end_time = datetime(2025,6,30,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[-1]) #datetime.now()
start_time = datetime(2025,5,7,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[0])
x_limit = (start_time, end_time)
date_string = f'{start_time.strftime("%d%m%H")}-{end_time.strftime("%d%m%H")}'

# Dynamically generate a colors dictionary with lists of colors
keys = list(data.keys())
colormap = plt.cm.get_cmap('tab10', len(keys))  # Use 'tab10' or any other colormap
colors = {key: [colormap(i)] for i, key in enumerate(keys)}  # Wrap each color in a list

# Use the FlexiblePlotter to create the plot
plotter.plot_multiple_variables_across_dfs_with_labels(
    data=data,
    variables=variable_groups,
    x_axis_format="datetime",
    x_limit=x_limit,
    x_axis_ticks_format="%d-%m %H",
    tick_rotation=90,
    interval=7*12, #hours of tick interval x
    grid=True,
    show_plot=False,
    y_limit=(0, 400),
    plot_types={'data_on':['line'], 'data_off_all':['line'], 'from_rpi':['line'],'rpi_data':['line'], 'osaprediction':['scatter']},
    colors=colors,
)
# output_path = os.path.join(os.path.dirname(input_path), 'awite_plots')
output_path = os.path.join(os.getcwd(), 'ClosedLooptest_tube_Real', 'plots_postprocess\\')

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the figure
#plt.savefig(os.path.join(output_path, f"1_{date_string}_u{plot_name}.png"), bbox_inches='tight', dpi=300)

# Log the output path
logging.info(f"Plot saved to {output_path}")
#------------------------------------------------------------------------------------------------------------# 

In [ ]:
#------------------------------------------------------------------------------------------------------------#
# PLOT
# Create interative plot! <<<<<<<<<<<<<<<<<<<<<<<<<
# Create an instance of FlexiblePlotter
plotter = FlexiblePlotter(default_figsize = (12, 4))

data = {"data_on_all": y_actual,
         "ekf": x_ekf,
         #"model": x_openloop,
         #"y_star": y_star_nom,
        #"y_ref": y_ref,
        # "osa_pred": osa_pred,
        }

# Dynamically create variable_groups as a dictionary
variable_groups = {key: ['qTot'] for key in data.keys()}
variable_groups['y_ref'] = ['Qch4_ref']
variable_groups['data_on_all'] = ['gas_rate']

# Calculate the x_limits
end_time = datetime(2025,6,30,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[-1]) #datetime.now()
start_time = datetime(2025,6,7,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[0])
x_limit = (start_time, end_time)
date_string = f'{start_time.strftime("%d%m%H")}-{end_time.strftime("%d%m%H")}'

# Dynamically generate a colors dictionary with lists of colors
keys = list(data.keys())
colormap = plt.cm.get_cmap('tab10', len(keys))  # Use 'tab10' or any other colormap
colors = {key: [colormap(i)] for i, key in enumerate(keys)}  # Wrap each color in a list

# Use the FlexiblePlotter to create the plot
plotter.plot_multiple_variables_across_dfs_with_labels(
    data=data,
    variables=variable_groups,
    x_axis_format="datetime",
    x_limit=x_limit,
    x_axis_ticks_format="%d-%m %H",
    tick_rotation=90,
    interval=7*12, #hours of tick interval x
    grid=True,
    show_plot=False,
    #y_limit=(10, 60),
    plot_types={'data_on':['line'], 'data_off_all':['line'], 'from_rpi':['line'],'rpi_data':['line'], 'osaprediction':['scatter']},
    colors=colors,
    legend_param={'loc': 'upper left'},
)
# output_path = os.path.join(os.path.dirname(input_path), 'awite_plots')
output_path = os.path.join(os.getcwd(), 'ClosedLooptest_tube_Real', 'plots_postprocess\\')

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the figure
#plt.savefig(os.path.join(output_path, f"1_{date_string}_qM{plot_name}.png"), bbox_inches='tight', dpi=300)

# Log the output path
logging.info(f"Plot saved to {output_path}")
#------------------------------------------------------------------------------------------------------------# 

In [ ]:
#------------------------------------------------------------------------------------------------------------#
# PLOT
# Create interative plot! <<<<<<<<<<<<<<<<<<<<<<<<<
# Create an instance of FlexiblePlotter
plotter = FlexiblePlotter(default_figsize = (12, 4))

data = {"data_on_all": y_actual,
         "ekf": x_ekf,
         #"model": x_openloop,
         #"y_star": y_star_nom,
        #"y_ref": y_ref,
         #"osa_pred": osa_pred,
        }

# Dynamically create variable_groups as a dictionary
variable_groups = {key: ['co2ch4_gb'] for key in data.keys()}
variable_groups['y_ref'] = ['co2ch4_ref']

# Calculate the x_limits
end_time = datetime(2025,6,30,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[-1]) #datetime.now()
start_time = datetime(2025,5,7,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[0])
x_limit = (start_time, end_time)
date_string = f'{start_time.strftime("%d%m%H")}-{end_time.strftime("%d%m%H")}'

# Dynamically generate a colors dictionary with lists of colors
keys = list(data.keys())
colormap = plt.cm.get_cmap('tab10', len(keys))  # Use 'tab10' or any other colormap
colors = {key: [colormap(i)] for i, key in enumerate(keys)}  # Wrap each color in a list

# Use the FlexiblePlotter to create the plot
plotter.plot_multiple_variables_across_dfs_with_labels(
    data=data,
    variables=variable_groups,
    x_axis_format="datetime",
    x_limit=x_limit,
    x_axis_ticks_format="%d-%m %H",
    tick_rotation=90,
    interval=7*12, #hours of tick interval x
    grid=True,
    show_plot=False,
    #y_limit=(0.5, 1.5),
    plot_types={'data_on':['line'], 'data_off_all':['line'], 'from_rpi':['line'],'rpi_data':['line'], 'osaprediction':['scatter']},
    colors=colors,
)
# output_path = os.path.join(os.path.dirname(input_path), 'awite_plots')
output_path = os.path.join(os.getcwd(), 'ClosedLooptest_tube_Real', 'plots_postprocess\\')

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the figure
#plt.savefig(os.path.join(output_path, f"1_{date_string}_co2ch4{plot_name}.png"), bbox_inches='tight', dpi=300)

# Log the output path
logging.info(f"Plot saved to {output_path}")
#------------------------------------------------------------------------------------------------------------# 

In [ ]:
#------------------------------------------------------------------------------------------------------------#
# PLOT
# Create interative plot! <<<<<<<<<<<<<<<<<<<<<<<<<
# Create an instance of FlexiblePlotter
plotter = FlexiblePlotter(default_figsize = (12, 4))

data = {
         "P_ekf": P_ekf,
        }

# Dynamically create variable_groups as a dictionary
variable_groups = {"P_ekf": P_ekf.keys()[1:].values.tolist()}

# Calculate the x_limits
end_time = datetime(2025,6,30,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[-1]) #datetime.now()
start_time = datetime(2025,5,13,10,0) #pd.to_datetime(x_actual['Timestamp'].iloc[0])
x_limit = (start_time, end_time)
date_string = f'{start_time.strftime("%d%m%H")}-{end_time.strftime("%d%m%H")}'

# Use the FlexiblePlotter to create the plot
plotter.plot_multiple_variables_across_dfs_with_labels(
    data=data,
    variables=variable_groups,
    x_axis_format="datetime",
    x_limit=x_limit,
    x_axis_ticks_format="%d-%m %H",
    tick_rotation=90,
    interval=7*12, #hours of tick interval x
    grid=True,
    show_plot=False,
    y_limit=(0, 0.5),
)
# output_path = os.path.join(os.path.dirname(input_path), 'awite_plots')
output_path = os.path.join(os.getcwd(), 'ClosedLooptest_tube_Real', 'plots_postprocess\\')

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the figure
#plt.savefig(os.path.join(output_path, f"1_{date_string}_cov{plot_name}.png"), bbox_inches='tight', dpi=300)

# Log the output path
logging.info(f"Plot saved to {output_path}")
#------------------------------------------------------------------------------------------------------------# 